# Data Cleaning: Companion Notebook

> Narrative companion for `src/my_mlops_project/pipelines/data_cleaning/`.
> The runnable logic lives in `nodes.py`; this notebook **shows the evidence and
> documents the decisions** behind it (feeds the report's data-cleaning section).

**Purpose:** turn the validated raw frame into the clean `03_primary` table, and
**justify how undocumented category codes are handled**. The central cleaning
decision for this dataset.

**Inputs:** `data/02_intermediate/validated_data.parquet` (from `data_quality`).
**Outputs:** `data/03_primary/cleaned_data.parquet`.

## Table of Contents
1. [Setup](#1-setup)
2. [Load inputs](#2-load-inputs)
3. [Explore: the undocumented categories](#3-explore-the-undocumented-categories)
4. [The cleaning decisions (and why)](#4-the-cleaning-decisions-and-why)
5. [Apply the cleaning](#5-apply-the-cleaning)
6. [Validate output](#6-validate-output)
7. [Notes for the report](#7-notes-for-the-report)

## 1. Setup

In [1]:
import sys
from pathlib import Path
import yaml
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Project root = two levels up from notebooks/. Put src/ on the path so we can
# import the production node (keeps this notebook in sync with the pipeline).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
DATA = PROJECT_ROOT / "data"

# Load the real cleaning policy from parameters.yml (single source of truth).
with open(PROJECT_ROOT / "conf" / "base" / "parameters.yml") as fh:
    params = yaml.safe_load(fh)["data_cleaning"]
params

{'id_column': 'ID',
 'target_column_raw': 'default payment next month',
 'target_column': 'default',
 'category_merges': {'EDUCATION': {'from': [0, 5, 6], 'to': 4},
  'MARRIAGE': {'from': [0], 'to': 3}}}

## 2. Load inputs

The quality-passed frame from the `data_quality` pipeline (values unchanged).

In [2]:
validated = pd.read_parquet(DATA / "02_intermediate" / "validated_data.parquet")
print("shape:", validated.shape)
print((validated.head()).to_string())

shape: (30000, 25)
   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6  BILL_AMT1  BILL_AMT2  BILL_AMT3  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  default payment next month
0   1      20000    2          2         1   24      2      2     -1     -1     -2     -2       3913       3102        689          0          0          0         0       689         0         0         0         0                           1
1   2     120000    2          2         2   26     -1      2      0      0      0      2       2682       1725       2682       3272       3455       3261         0      1000      1000      1000         0      2000                           1
2   3      90000    2          2         2   34      0      0      0      0      0      0      29239      14027      13559      14331      14948      15549      1518      1500      1000      1000      1000      5000                           0
3   4

## 3. Explore: the undocumented categories

The dataset's **data dictionary** documents these codes:

| Column | Documented codes |
|---|---|
| `EDUCATION` | 1 = grad school, 2 = university, 3 = high school, 4 = other |
| `MARRIAGE` | 1 = married, 2 = single, 3 = other |
| `PAY_*` | −1 = paid duly, 1...9 = months of delay |

But the **actual data contains extra codes the dictionary never explains**. Before
deciding what to do, we quantify how common each undocumented code is. The
decision depends entirely on this.

In [3]:
n = len(validated)

for col, documented in [("EDUCATION", {1, 2, 3, 4}), ("MARRIAGE", {1, 2, 3})]:
    vc = validated[col].value_counts().sort_index()
    undoc = [v for v in vc.index if v not in documented]
    rows = int(validated[col].isin(undoc).sum())
    print(f"{col}: {dict(vc)}")
    print(f"   undocumented {undoc} -> {rows} rows ({100*rows/n:.2f}%)\n")

pay_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
documented_pay = set(range(-1, 10))
undoc_pay = sorted({v for c in pay_cols for v in validated[c].unique() if v not in documented_pay})
rows_pay = int(validated[pay_cols].isin(undoc_pay).any(axis=1).sum())
print(f"PAY_* undocumented codes {undoc_pay} -> {rows_pay} rows have >=1 ({100*rows_pay/n:.1f}%)")

EDUCATION: {0: np.int64(14), 1: np.int64(10585), 2: np.int64(14030), 3: np.int64(4917), 4: np.int64(123), 5: np.int64(280), 6: np.int64(51)}
   undocumented [0, 5, 6] -> 345 rows (1.15%)

MARRIAGE: {0: np.int64(54), 1: np.int64(13659), 2: np.int64(15964), 3: np.int64(323)}
   undocumented [0] -> 54 rows (0.18%)

PAY_* undocumented codes [np.int64(-2)] -> 6561 rows have >=1 (21.9%)


## 4. The cleaning decisions (and why)

The exploration above shows the undocumented codes are **not all alike**, so we do
**not** treat them the same way:

| Column | Undocumented | Frequency | **Decision** | Why |
|---|---|---|---|---|
| `EDUCATION` | 0, 5, 6 | ~1.15% | **merge to 4 (other)** | rare and unexplained; "other" is the honest bucket for unknown levels. Dropping the rows would discard real customers (and their default outcomes) over one odd code. |
| `MARRIAGE` | 0 | ~0.18% | **merge to 3 (other)** | same reasoning, even rarer. |
| `PAY_*` | −2 (and 0) | **~22%** | **KEEP unchanged** | −2 is *common* and *meaningful*. It represents "no credit used that month". The PAY columns are an **ordinal scale** (−2 < −1 < 0 < 1 < ...); merging or dropping −2 would destroy ~22% of genuine repayment signal. |

**Other steps** (mechanical, not judgement calls): drop the `ID` row identifier
(arbitrary key, no predictive value), rename the spaced target
`default payment next month` to `default`, drop exact duplicate rows, and coerce to
integer dtypes.

> The takeaway worth stating in the report: *"undocumented" is not the same as
> "invalid".* We quantified each case and only consolidated the rare, meaningless
> codes. A deliberate, evidence-based decision rather than a blanket rule.

## 5. Apply the cleaning

We call the **production node** (`nodes.clean_credit_data`) so this notebook and the Kedro pipeline can never drift apart.

In [4]:
from my_mlops_project.pipelines.data_cleaning.nodes import clean_credit_data

cleaned = clean_credit_data(validated, params)

print("before:", validated.shape, "-> after:", cleaned.shape)
print((cleaned.head()).to_string())

data_cleaning: dropped 'ID', renamed target -> 'default', merged ['EDUCATION', 'MARRIAGE'] undocumented codes, removed 35 duplicate rows -> (29965, 24)
before: (30000, 25) -> after: (29965, 24)
   LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  PAY_5  PAY_6  BILL_AMT1  BILL_AMT2  BILL_AMT3  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  default
0      20000    2          2         1   24      2      2     -1     -1     -2     -2       3913       3102        689          0          0          0         0       689         0         0         0         0        1
1     120000    2          2         2   26     -1      2      0      0      0      2       2682       1725       2682       3272       3455       3261         0      1000      1000      1000         0      2000        1
2      90000    2          2         2   34      0      0      0      0      0      0      29239      14027      13559      14331      14948   

In [5]:
# Confirm the decisions took effect.
print("EDUCATION values:", sorted(cleaned["EDUCATION"].unique()))   # -> [1,2,3,4]
print("MARRIAGE  values:", sorted(cleaned["MARRIAGE"].unique()))    # -> [1,2,3]
print("PAY_0 still has -2:", -2 in cleaned["PAY_0"].values)         # -> True (preserved)
print("ID dropped:", "ID" not in cleaned.columns, "| target renamed:", "default" in cleaned.columns)

EDUCATION values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
MARRIAGE  values: [np.int64(1), np.int64(2), np.int64(3)]
PAY_0 still has -2: True
ID dropped: True | target renamed: True


## 6. Validate output

Assertions that protect the downstream feature-engineering pipeline.

In [6]:
assert "ID" not in cleaned.columns, "ID should be dropped"
assert "default" in cleaned.columns, "target should be renamed to 'default'"
assert set(cleaned["EDUCATION"].unique()).issubset({1, 2, 3, 4}), "EDUCATION not consolidated"
assert set(cleaned["MARRIAGE"].unique()).issubset({1, 2, 3}), "MARRIAGE not consolidated"
assert -2 in cleaned["PAY_0"].values, "PAY -2 must be preserved"
print("All cleaning assertions passed.")

All cleaning assertions passed.


## 7. Notes for the report

> For [`../report/REPORT_OUTLINE.md`](../report/REPORT_OUTLINE.md) section 3 (data exploration / cleaning).

- **Undocumented category codes were detected in the `data_quality` step and handled
  deliberately**, not with a blanket rule:
  - `EDUCATION {0,5,6}` (~1.15%) and `MARRIAGE {0}` (~0.18%) to merged into the
    documented "other" category (rare + unexplained).
  - `PAY_* −2` (~22% of rows) to **kept**, because it is common and meaningful
    (no credit used); the PAY columns are an ordinal repayment scale.
- **35 exact-duplicate customers** were removed after dropping the `ID` key
  (30,000 to 29,965 rows).
- **Production risk / mitigation:** the merge maps live in `parameters.yml`, so if a
  future batch introduces a new undocumented code, the `data_quality` value-set
  expectation flags it and we extend the merge map. The policy is config, not code.